## Imports & Setup

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from torch.utils.data import DataLoader, Subset

from dataset_classes import ISO_NE, SH_Dataset
from models_with_temporal_graph import TR_GNN_MultiScale

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Device: {device}")

🖥️  Device: cuda


## Sensitivity Config Dict

In [2]:
# ============================================================
# Shared sensitivity configs (same grid for both datasets)
# ============================================================
sensitivity_configs = {
    # GCN Depth
    "gcn_depth_1": {"GCN_Layer": 1,  "hidden_dim":  64, "kernel_size":  7, "dilation": 3},
    "gcn_depth_2": {"GCN_Layer": 2,  "hidden_dim":  64, "kernel_size":  7, "dilation": 3},
    "gcn_depth_3": {"GCN_Layer": 3,  "hidden_dim":  64, "kernel_size":  7, "dilation": 3},
    "gcn_depth_7": {"GCN_Layer": 7,  "hidden_dim":  64, "kernel_size":  7, "dilation": 3},
    # Hidden Size
    "hidden_32":   {"GCN_Layer": 5,  "hidden_dim":  32, "kernel_size":  7, "dilation": 3},
    "hidden_128":  {"GCN_Layer": 5,  "hidden_dim": 128, "kernel_size":  7, "dilation": 3},
    "hidden_256":  {"GCN_Layer": 5,  "hidden_dim": 256, "kernel_size":  7, "dilation": 3},
    # Kernel Size
    "kernel_3":    {"GCN_Layer": 5,  "hidden_dim":  64, "kernel_size":  3, "dilation": 3},
    "kernel_5":    {"GCN_Layer": 5,  "hidden_dim":  64, "kernel_size":  5, "dilation": 3},
    "kernel_11":   {"GCN_Layer": 5,  "hidden_dim":  64, "kernel_size": 11, "dilation": 3},
    # Dilation
    "dilation_1":  {"GCN_Layer": 5,  "hidden_dim":  64, "kernel_size":  7, "dilation": 1},
    "dilation_2":  {"GCN_Layer": 5,  "hidden_dim":  64, "kernel_size":  7, "dilation": 2},
    "dilation_5":  {"GCN_Layer": 5,  "hidden_dim":  64, "kernel_size":  7, "dilation": 5},
}

print(f"Configs loaded: {len(sensitivity_configs)}")

Configs loaded: 13


## Evaluation Function (scaled + unscaled)

In [3]:
# ============================================================
# Shared evaluation helper — scaled + unscaled metrics
# ============================================================
def evaluate_model(model, test_loader, dataset, device):
    """
    Returns scaled and unscaled MSE, MAE, R², plus the mean batch test loss.
    Works for any dataset that exposes .scaler, .feature_names, .target_idx.
    """
    criterion = nn.MSELoss()
    model.eval()

    preds_all, trues_all = [], []
    total_loss = 0.0

    with torch.no_grad():
        for X, Y in test_loader:
            X, Y = X.to(device), Y.to(device)
            pred, _ = model(X)                    # unpack (pred, A)
            total_loss += criterion(pred, Y).item()
            preds_all.append(pred.cpu().numpy())
            trues_all.append(Y.cpu().numpy())

    test_loss_scaled = total_loss / len(test_loader)

    preds_flat = np.concatenate(preds_all).flatten()
    trues_flat = np.concatenate(trues_all).flatten()

    # ── Scaled ───────────────────────────────────────────────
    mse_scaled = mean_squared_error(trues_flat, preds_flat)
    mae_scaled = mean_absolute_error(trues_flat, preds_flat)
    r2_scaled  = r2_score(trues_flat, preds_flat)

    # ── Unscaled (inverse transform target column only) ──────
    n           = len(preds_flat)
    n_features  = len(dataset.feature_names)
    target_col  = dataset.target_idx

    dummy_preds = np.zeros((n, n_features), dtype=np.float32)
    dummy_trues = np.zeros((n, n_features), dtype=np.float32)
    dummy_preds[:, target_col] = preds_flat
    dummy_trues[:, target_col] = trues_flat

    preds_unscaled = dataset.scaler.inverse_transform(dummy_preds)[:, target_col]
    trues_unscaled = dataset.scaler.inverse_transform(dummy_trues)[:, target_col]

    mse_unscaled = mean_squared_error(trues_unscaled, preds_unscaled)
    mae_unscaled = mean_absolute_error(trues_unscaled, preds_unscaled)
    r2_unscaled  = r2_score(trues_unscaled, preds_unscaled)

    return {
        "test_loss_scaled" : test_loss_scaled,
        "mse_scaled"       : mse_scaled,
        "mae_scaled"       : mae_scaled,
        "r2_scaled"        : r2_scaled,
        "mse_unscaled"     : mse_unscaled,
        "mae_unscaled"     : mae_unscaled,
        "r2_unscaled"      : r2_unscaled,
    }

## Evaluation Loop

In [ ]:
# ============================================================
# Loads every .pth in a folder and evaluates
# ============================================================
def run_evaluation_loop(sensitivity_configs, base_hparams, model_dir,
                        dataset, test_loader, device, tag=""):
    """
    Iterates over sensitivity_configs, loads the checkpoint from
    model_dir/{cfg_name}_best_model.pth, evaluates on test_loader,
    and returns a results dict keyed by cfg_name.
    Failures are caught per-model so one bad checkpoint never aborts the run.
    """
    results = {}

    for cfg_name, cfg_overrides in sensitivity_configs.items():
        model_path = os.path.join(model_dir, f"{cfg_name}_best_model.pth")

        if not os.path.exists(model_path):
            print(f"⚠️  [{tag}] SKIPPED {cfg_name} — not found: {model_path}")
            results[cfg_name] = {"status": "NOT_FOUND", **cfg_overrides}
            continue

        try:
            hp = {**base_hparams, **cfg_overrides}

            model = TR_GNN_MultiScale(
                N=hp["N"], T_in=hp["T_in"], T_out=hp["T_out"], d=hp["d"],
                hidden_dim=hp["hidden_dim"], GCN_Layer=hp["GCN_Layer"],
                dropout_gcn=hp["dropout_gcn"], dropout_temporal=hp["dropout_temporal"],
                kernel_size=hp["kernel_size"], dilation=hp["dilation"],
            ).to(device)

            ckpt = torch.load(model_path, map_location=device)
            model.load_state_dict(ckpt["model_state_dict"])
            model.eval()

            metrics = evaluate_model(model, test_loader, dataset, device)
            n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

            results[cfg_name] = {
                "status"          : "OK",
                "val_loss_ckpt"   : ckpt.get("val_loss", float("nan")),
                "n_params"        : n_params,
                **cfg_overrides,
                **metrics,
            }

            print(
                f"✅ [{tag}] {cfg_name:<15} | "
                f"val(ckpt)={results[cfg_name]['val_loss_ckpt']:.4f} | "
                f"loss_scaled={metrics['test_loss_scaled']:.4f} | "
                f"MSE_u={metrics['mse_unscaled']:.2f}  "
                f"MAE_u={metrics['mae_unscaled']:.2f}  "
                f"R²_u={metrics['r2_unscaled']:.4f}"
            )

        except Exception as exc:
            import traceback
            print(f"❌ [{tag}] FAILED {cfg_name}: {exc}")
            traceback.print_exc()
            results[cfg_name] = {"status": "FAILED", "error": str(exc), **cfg_overrides}

    ok = sum(1 for r in results.values() if r.get("status") == "OK")
    print(f"\n🏁 [{tag}] Evaluated {ok} / {len(sensitivity_configs)} models.\n")
    return results

## Grouped Summary Function

In [5]:
# ============================================================
# Shared grouped summary
# ============================================================

_CATEGORY_PREFIXES = {
    "GCN Depth"        : "gcn_depth_",
    "Hidden Dimension" : "hidden_",
    "Kernel Size"      : "kernel_",
    "Dilation"         : "dilation_",
}

_CATEGORY_COL = {
    "GCN Depth"        : "GCN Depth",
    "Hidden Dimension" : "Hidden Dim",
    "Kernel Size"      : "Kernel Size",
    "Dilation"         : "Dilation",
}

def build_grouped_summary(test_results, save_dir, tag=""):
    rows = []
    for cfg_name, r in test_results.items():
        if r.get("status") != "OK":
            continue
        rows.append({
            "Config"       : cfg_name,
            "GCN Depth"    : r.get("GCN_Layer"),
            "Hidden Dim"   : r.get("hidden_dim"),
            "Kernel Size"  : r.get("kernel_size"),
            "Dilation"     : r.get("dilation"),
            "MSE_scaled"   : r.get("mse_scaled"),
            "MSE_unscaled" : r.get("mse_unscaled"),
            "MAE_scaled"   : r.get("mae_scaled"),
            "MAE_unscaled" : r.get("mae_unscaled"),
            "R2_scaled"    : r.get("r2_scaled"),
            "R2_unscaled"  : r.get("r2_unscaled"),
        })

    if not rows:
        print(f"[{tag}] No successful results to summarise.")
        return None

    df_base = pd.DataFrame(rows)

    def fmt(scaled, unscaled):
        if pd.isna(unscaled):
            return f"{scaled:.4f}"
        return f"{scaled:.4f} / {unscaled:.4f}"

    final_rows = []
    for cat_label, prefix in _CATEGORY_PREFIXES.items():
        col_key = _CATEGORY_COL[cat_label]

        # Filter to only configs belonging to this sensitivity group
        cat_df = df_base[df_base["Config"].str.startswith(prefix)].copy()

        if cat_df.empty:
            print(f"  ⚠️  [{tag}] No configs found for category '{cat_label}' (prefix='{prefix}')")
            continue

        best_val = cat_df.loc[cat_df["MSE_scaled"].idxmin(), col_key]

        for _, row in cat_df.sort_values(col_key).iterrows():
            final_rows.append({
                "Category"                : cat_label,
                "Value"                   : row[col_key],
                "MSE (Scaled / Unscaled)" : fmt(row["MSE_scaled"],  row["MSE_unscaled"]),
                "MAE (Scaled / Unscaled)" : fmt(row["MAE_scaled"],  row["MAE_unscaled"]),
                "R² (Scaled / Unscaled)"  : fmt(row["R2_scaled"],   row["R2_unscaled"]),
                "Remarks"                 : "✅ Best" if row[col_key] == best_val else "",
            })

    df_sensitivity = pd.DataFrame(final_rows).set_index(["Category", "Value"])

    print(f"\n{'='*120}")
    print(f"SENSITIVITY ANALYSIS — {tag} — GROUPED TEST RESULTS")
    print(f"{'='*120}")
    print(df_sensitivity.to_string())

    os.makedirs(save_dir, exist_ok=True)
    out_path = os.path.join(save_dir, f"sensitivity_grouped_results_{tag}.csv")
    df_sensitivity.to_csv(out_path)
    print(f"\n💾 Saved → {out_path}")

    return df_sensitivity

## ISO-NE Dataset & Test Split

In [6]:
# ============================================================
# ISO-NE — Dataset Setup & Test Split (load_forecast logic)
# ============================================================
try:
    ISONE_MODEL_DIR = "models_ISO_NE"

    dataset_isone = ISO_NE(
        csv_path="selected_data_ISONE.csv",
        T_in=72,
        T_out=240,
        lag_hours=[1, 24],
        rolling_windows=[24],
    )

    total_len_isone       = len(dataset_isone.df_numeric)
    train_split_isone     = int(0.6 * total_len_isone)
    val_split_isone       = int(0.8 * total_len_isone)

    scaler_isone = StandardScaler()
    scaler_isone.fit(dataset_isone.df_numeric.iloc[:train_split_isone].values.astype(np.float32))
    dataset_isone.apply_scaler(scaler_isone)
    dataset_isone.scaler = scaler_isone

    effective_len_isone = len(dataset_isone)
    train_end_isone     = min(train_split_isone - dataset_isone.T_in, effective_len_isone)
    val_end_isone       = min(val_split_isone   - dataset_isone.T_in, effective_len_isone)
    test_idx_isone      = range(val_end_isone, effective_len_isone)

    test_loader_isone = DataLoader(
        Subset(dataset_isone, test_idx_isone),
        batch_size=32, shuffle=False,
    )

    BASE_HPARAMS_ISONE = dict(
        N=dataset_isone.N, T_in=72, T_out=240, d=32,
        dropout_gcn=0.2, dropout_temporal=0.2,
        GCN_Layer=5, hidden_dim=64, kernel_size=7, dilation=3,
    )

    print(f"✅ ISO-NE ready | Raw rows: {total_len_isone} | "
          f"Test samples: {len(test_idx_isone)} | "
          f"Test batches: {len(test_loader_isone)} | "
          f"Model dir: {ISONE_MODEL_DIR}/")

    _isone_dataset_ok = True

except Exception as _e:
    print(f"❌ ISO-NE dataset setup FAILED: {_e}")
    _isone_dataset_ok = False

Loaded dataset with 16 features (target=demand), total rows=103752
✅ ISO-NE ready | Raw rows: 103752 | Test samples: 20511 | Test batches: 641 | Model dir: models_ISO_NE/


## ISO-NE Evaluation Loop

In [7]:
# ============================================================
# ISO-NE — Run evaluation 
# ============================================================
if _isone_dataset_ok:
    try:
        test_results_isone = run_evaluation_loop(
            sensitivity_configs = sensitivity_configs,
            base_hparams        = BASE_HPARAMS_ISONE,
            model_dir           = ISONE_MODEL_DIR,
            dataset             = dataset_isone,
            test_loader         = test_loader_isone,
            device              = device,
            tag                 = "ISO-NE",
        )
        _isone_eval_ok = True
    except Exception as _e:
        import traceback
        print(f"❌ ISO-NE evaluation loop FAILED: {_e}")
        traceback.print_exc()
        test_results_isone = {}
        _isone_eval_ok = False
else:
    print("⏭️  ISO-NE evaluation skipped (dataset setup failed).")
    test_results_isone = {}
    _isone_eval_ok = False

✅ [ISO-NE] gcn_depth_1     | val(ckpt)=0.1845 | loss_scaled=0.1671 | MSE_u=1392138.38  MAE_u=843.71  R²_u=0.8176
✅ [ISO-NE] gcn_depth_2     | val(ckpt)=0.1841 | loss_scaled=0.1663 | MSE_u=1384742.38  MAE_u=835.54  R²_u=0.8186
✅ [ISO-NE] gcn_depth_3     | val(ckpt)=0.1951 | loss_scaled=0.1739 | MSE_u=1448190.75  MAE_u=862.91  R²_u=0.8103
✅ [ISO-NE] gcn_depth_7     | val(ckpt)=0.1919 | loss_scaled=0.1698 | MSE_u=1414661.62  MAE_u=835.85  R²_u=0.8147
✅ [ISO-NE] hidden_32       | val(ckpt)=0.1962 | loss_scaled=0.1770 | MSE_u=1474173.88  MAE_u=885.59  R²_u=0.8069
✅ [ISO-NE] hidden_128      | val(ckpt)=0.1910 | loss_scaled=0.1696 | MSE_u=1412474.25  MAE_u=838.33  R²_u=0.8150
✅ [ISO-NE] hidden_256      | val(ckpt)=0.1923 | loss_scaled=0.1705 | MSE_u=1420490.00  MAE_u=846.91  R²_u=0.8139
✅ [ISO-NE] kernel_3        | val(ckpt)=0.1846 | loss_scaled=0.1681 | MSE_u=1400278.12  MAE_u=845.09  R²_u=0.8166
✅ [ISO-NE] kernel_5        | val(ckpt)=0.1972 | loss_scaled=0.1772 | MSE_u=1475601.25  MAE_u=875

## ISO-NE Grouped Summary

In [8]:
# ============================================================
# ISO-NE — Grouped summary table
# ============================================================
if _isone_eval_ok and test_results_isone:
    df_summary_isone = build_grouped_summary(
        test_results = test_results_isone,
        save_dir     = "models_ISO_NE",
        tag          = "ISO-NE",
    )
else:
    print("⏭️  ISO-NE summary skipped.")
    df_summary_isone = None


SENSITIVITY ANALYSIS — ISO-NE — GROUPED TEST RESULTS
                       MSE (Scaled / Unscaled) MAE (Scaled / Unscaled) R² (Scaled / Unscaled) Remarks
Category         Value                                                                               
GCN Depth        1       0.1671 / 1392138.3750       0.2923 / 843.7139        0.8176 / 0.8176        
                 2       0.1663 / 1384742.3750       0.2895 / 835.5432        0.8186 / 0.8186  ✅ Best
                 3       0.1739 / 1448190.7500       0.2990 / 862.9094        0.8103 / 0.8103        
                 7       0.1698 / 1414661.6250       0.2896 / 835.8508        0.8147 / 0.8147        
Hidden Dimension 32      0.1770 / 1474173.8750       0.3069 / 885.5895        0.8069 / 0.8069        
                 128     0.1696 / 1412474.2500       0.2905 / 838.3305        0.8150 / 0.8150  ✅ Best
                 256     0.1705 / 1420490.0000       0.2935 / 846.9066        0.8139 / 0.8139        
Kernel Size      3       0.1

## ISO-NE Raw Results Table

In [9]:
# ============================================================
# ISO-NE — Full flat results table (all metrics, all models)
# ============================================================
if _isone_eval_ok and test_results_isone:
    _rows = []
    for cfg_name, r in test_results_isone.items():
        if r.get("status") != "OK":
            continue
        _rows.append({
            "Config"             : cfg_name,
            "GCN_Layer"          : r.get("GCN_Layer"),
            "hidden_dim"         : r.get("hidden_dim"),
            "kernel_size"        : r.get("kernel_size"),
            "dilation"           : r.get("dilation"),
            "# Params"           : r.get("n_params"),
            "Val Loss (ckpt)"    : r.get("val_loss_ckpt"),
            "Test Loss (scaled)" : r.get("test_loss_scaled"),
            "MSE  (scaled)"      : r.get("mse_scaled"),
            "MAE  (scaled)"      : r.get("mae_scaled"),
            "R²   (scaled)"      : r.get("r2_scaled"),
            "MSE  (unscaled)"    : r.get("mse_unscaled"),
            "MAE  (unscaled)"    : r.get("mae_unscaled"),
            "R²   (unscaled)"    : r.get("r2_unscaled"),
        })

    df_flat_isone = pd.DataFrame(_rows).sort_values("Test Loss (scaled)")
    pd.set_option("display.float_format", "{:.4f}".format)
    pd.set_option("display.max_columns", 20)
    pd.set_option("display.width", 220)

    print("\n" + "="*120)
    print("ISO-NE — FULL FLAT TEST RESULTS")
    print("="*120)
    print(df_flat_isone.to_string(index=False))

    df_flat_isone.to_csv(os.path.join("models_ISO_NE", "sensitivity_test_results_ISONE.csv"), index=False)
    print(f"\n💾 Saved → models_ISO_NE/sensitivity_test_results_ISONE.csv")
else:
    print("⏭️  ISO-NE flat table skipped.")


ISO-NE — FULL FLAT TEST RESULTS
     Config  GCN_Layer  hidden_dim  kernel_size  dilation  # Params  Val Loss (ckpt)  Test Loss (scaled)  MSE  (scaled)  MAE  (scaled)  R²   (scaled)  MSE  (unscaled)  MAE  (unscaled)  R²   (unscaled)
gcn_depth_2          2          64            7         3    690272           0.1841              0.1663         0.1663         0.2895         0.8186     1384742.3750         835.5432           0.8186
gcn_depth_1          1          64            7         3    686112           0.1845              0.1671         0.1671         0.2923         0.8176     1392138.3750         843.7139           0.8176
   kernel_3          5          64            3         3    698656           0.1846              0.1681         0.1681         0.2928         0.8166     1400278.1250         845.0906           0.8166
 dilation_1          5          64            7         1    702752           0.1854              0.1684         0.1684         0.2942         0.8163     1402640.6

## SH Dataset & Test Split

In [10]:
# ============================================================
# SH — Dataset Setup & Test Split (load_forecast_sh logic)
# ============================================================
try:
    SH_MODEL_DIR = "models_SH"

    dataset_sh = SH_Dataset(
        csv_path="shanghai.csv",
        T_in=72,
        T_out=240,
        lag_hours=[1, 12, 24, 168],
        rolling_windows=[12, 24],
    )

    total_len_sh       = len(dataset_sh.df_numeric)
    train_split_sh     = int(0.6 * total_len_sh)
    val_split_sh       = int(0.8 * total_len_sh)

    scaler_sh = StandardScaler()
    scaler_sh.fit(dataset_sh.df_numeric.iloc[:train_split_sh].values.astype(np.float32))
    dataset_sh.apply_scaler(scaler_sh)
    dataset_sh.scaler = scaler_sh

    effective_len_sh = len(dataset_sh)
    train_end_sh     = min(train_split_sh - dataset_sh.T_in, effective_len_sh)
    val_end_sh       = min(val_split_sh   - dataset_sh.T_in, effective_len_sh)
    test_idx_sh      = range(val_end_sh, effective_len_sh)

    test_loader_sh = DataLoader(
        Subset(dataset_sh, test_idx_sh),
        batch_size=32, shuffle=False,
    )

    BASE_HPARAMS_SH = dict(
        N=dataset_sh.N, T_in=72, T_out=240, d=32,
        dropout_gcn=0.2, dropout_temporal=0.2,
        GCN_Layer=5, hidden_dim=64, kernel_size=7, dilation=3,
    )

    print(f"✅ SH ready       | Raw rows: {total_len_sh} | "
          f"Test samples: {len(test_idx_sh)} | "
          f"Test batches: {len(test_loader_sh)} | "
          f"Model dir: {SH_MODEL_DIR}/")

    _sh_dataset_ok = True

except Exception as _e:
    print(f"❌ SH dataset setup FAILED: {_e}")
    _sh_dataset_ok = False

Loaded dataset with 27 features (target=load), total rows=31314
✅ SH ready       | Raw rows: 31314 | Test samples: 6023 | Test batches: 189 | Model dir: models_SH/


## SH Evaluation Loop

In [11]:
# ============================================================
# SH — Run evaluation 
# ============================================================
if _sh_dataset_ok:
    try:
        test_results_sh = run_evaluation_loop(
            sensitivity_configs = sensitivity_configs,
            base_hparams        = BASE_HPARAMS_SH,
            model_dir           = SH_MODEL_DIR,
            dataset             = dataset_sh,
            test_loader         = test_loader_sh,
            device              = device,
            tag                 = "SH",
        )
        _sh_eval_ok = True
    except Exception as _e:
        import traceback
        print(f"❌ SH evaluation loop FAILED: {_e}")
        traceback.print_exc()
        test_results_sh = {}
        _sh_eval_ok = False
else:
    print("⏭️  SH evaluation skipped (dataset setup failed).")
    test_results_sh = {}
    _sh_eval_ok = False

✅ [SH] gcn_depth_1     | val(ckpt)=nan | loss_scaled=0.2168 | MSE_u=3707036.00  MAE_u=1353.36  R²_u=0.7217
✅ [SH] gcn_depth_2     | val(ckpt)=nan | loss_scaled=0.2151 | MSE_u=3676973.25  MAE_u=1352.27  R²_u=0.7239
✅ [SH] gcn_depth_3     | val(ckpt)=nan | loss_scaled=0.2129 | MSE_u=3637810.25  MAE_u=1343.18  R²_u=0.7269
✅ [SH] gcn_depth_7     | val(ckpt)=nan | loss_scaled=0.2152 | MSE_u=3678071.50  MAE_u=1356.82  R²_u=0.7238
✅ [SH] hidden_32       | val(ckpt)=nan | loss_scaled=0.2113 | MSE_u=3607457.50  MAE_u=1345.83  R²_u=0.7291
✅ [SH] hidden_128      | val(ckpt)=nan | loss_scaled=0.2198 | MSE_u=3752388.50  MAE_u=1367.91  R²_u=0.7183
✅ [SH] hidden_256      | val(ckpt)=nan | loss_scaled=0.2173 | MSE_u=3700755.75  MAE_u=1362.07  R²_u=0.7221
✅ [SH] kernel_3        | val(ckpt)=nan | loss_scaled=0.2033 | MSE_u=3464688.00  MAE_u=1318.35  R²_u=0.7399
✅ [SH] kernel_5        | val(ckpt)=nan | loss_scaled=0.2088 | MSE_u=3562061.25  MAE_u=1333.56  R²_u=0.7326
✅ [SH] kernel_11       | val(ckpt)=na

## SH Grouped Summary

In [12]:
# ============================================================
# SH — Grouped summary table
# ============================================================
if _sh_eval_ok and test_results_sh:
    df_summary_sh = build_grouped_summary(
        test_results = test_results_sh,
        save_dir     = "models_SH",
        tag          = "SH",
    )
else:
    print("⏭️  SH summary skipped.")
    df_summary_sh = None


SENSITIVITY ANALYSIS — SH — GROUPED TEST RESULTS
                       MSE (Scaled / Unscaled) MAE (Scaled / Unscaled) R² (Scaled / Unscaled) Remarks
Category         Value                                                                               
GCN Depth        1       0.2167 / 3707036.0000      0.3272 / 1353.3564        0.7217 / 0.7217        
                 2       0.2149 / 3676973.2500      0.3270 / 1352.2661        0.7239 / 0.7239        
                 3       0.2127 / 3637810.2500      0.3248 / 1343.1763        0.7269 / 0.7269  ✅ Best
                 7       0.2150 / 3678071.5000      0.3281 / 1356.8217        0.7238 / 0.7238        
Hidden Dimension 32      0.2109 / 3607457.5000      0.3254 / 1345.8256        0.7291 / 0.7291  ✅ Best
                 128     0.2194 / 3752388.5000      0.3307 / 1367.9078        0.7183 / 0.7183        
                 256     0.2163 / 3700755.7500      0.3293 / 1362.0699        0.7221 / 0.7221        
Kernel Size      3       0.2025 

## SH Raw Results Table

In [14]:
# ============================================================
# SH — Full flat results table (all metrics, all models)
# ============================================================
if _sh_eval_ok and test_results_sh:
    _rows = []
    for cfg_name, r in test_results_sh.items():
        if r.get("status") != "OK":
            continue
        _rows.append({
            "Config"             : cfg_name,
            "GCN_Layer"          : r.get("GCN_Layer"),
            "hidden_dim"         : r.get("hidden_dim"),
            "kernel_size"        : r.get("kernel_size"),
            "dilation"           : r.get("dilation"),
            "# Params"           : r.get("n_params"),
            "Val Loss (ckpt)"    : r.get("val_loss_ckpt"),
            "Test Loss (scaled)" : r.get("test_loss_scaled"),
            "MSE  (scaled)"      : r.get("mse_scaled"),
            "MAE  (scaled)"      : r.get("mae_scaled"),
            "R²   (scaled)"      : r.get("r2_scaled"),
            "MSE  (unscaled)"    : r.get("mse_unscaled"),
            "MAE  (unscaled)"    : r.get("mae_unscaled"),
            "R²   (unscaled)"    : r.get("r2_unscaled"),
        })

    df_flat_sh = pd.DataFrame(_rows).sort_values("Test Loss (scaled)")
    print("\n" + "="*120)
    print("SH — FULL FLAT TEST RESULTS")
    print("="*120)
    print(df_flat_sh.to_string(index=False))

    df_flat_sh.to_csv(os.path.join("models_SH", "sensitivity_test_results_SH.csv"), index=False)
    print(f"\n💾 Saved → models_SH/sensitivity_test_results_SH.csv")
else:
    print("⏭️  SH flat table skipped.")


SH — FULL FLAT TEST RESULTS
     Config  GCN_Layer  hidden_dim  kernel_size  dilation  # Params  Val Loss (ckpt)  Test Loss (scaled)  MSE  (scaled)  MAE  (scaled)  R²   (scaled)  MSE  (unscaled)  MAE  (unscaled)  R²   (unscaled)
   kernel_3          5          64            3         3   1061920              NaN              0.2033         0.2025         0.3188         0.7399     3464688.0000        1318.3513           0.7399
 dilation_2          5          64            7         2   1068832              NaN              0.2046         0.2043         0.3194         0.7376     3494941.7500        1321.1353           0.7376
 dilation_1          5          64            7         1   1068832              NaN              0.2069         0.2066         0.3221         0.7346     3534166.5000        1332.1505           0.7346
   kernel_5          5          64            5         3   1065376              NaN              0.2088         0.2082         0.3224         0.7326     3562061.2500 